# MOT20 Tracking Pipeline

End-to-end pipeline: data preparation → YOLO training → evaluation.

**Data split:**
| Split | Sequences | Purpose |
|---|---|---|
| train | MOT20-01, MOT20-02 | YOLO training |
| val | MOT20-03 | Monitored during training |
| labeled_test | MOT20-05 | Final evaluation only — do not tune on this |
| test (no GT) | MOT20-04, 06, 07, 08 | Inference / visual inspection |

## 0. Environment setup

Set `ENVIRONMENT` to `"colab"` or `"local"` and fill in your paths.

In [ ]:
!git clone

In [ ]:
# ── Choose your environment ────────────────────────────────────────────────
ENVIRONMENT = "colab"   # "local" | "colab"

if ENVIRONMENT == "colab":
    from google.colab import drive
    drive.mount("/content/drive")

    # Root folder that contains MOT20/, mot20_mvs/, train_mot20/
    DATA_ROOT = "/content/drive/MyDrive/ComputerVision/data/"
    # Path to cloned / uploaded repo on Drive (must contain tracking/src/)
    REPO_ROOT = "/content/drive/MyDrive/ComputerVision/"
    # Where YOLO saves training runs
    RUNS_DIR  = "/content/drive/MyDrive/ComputerVision/tracking/runs/detect"
    DEVICE    = "cuda"

else:
    # Paths are relative to the tracking/ folder where this notebook lives
    DATA_ROOT = "../data/"
    REPO_ROOT = "../"
    RUNS_DIR  = "runs/detect"   # saved inside tracking/runs/detect
    DEVICE    = "mps"           # "mps" (Apple) | "cuda" | "cpu"

In [ ]:
import sys, os

sys.path.insert(0, os.path.join(REPO_ROOT, "tracking", "src"))

from tracking_51 import (
    configure, Dataset,
    get_sequence_view,
    tracking_yolo, calc_metrics, save_results,
)
from tracking import train_detector
import fiftyone as fo

# Apply path configuration — must be called before any Dataset/tracking function
configure(data_root=DATA_ROOT)

---
## 1. Data preparation

Run once. Creates mp4 videos, builds the FiftyOne dataset, adds GT annotations,
and exports YOLO-format labels + images to `data/train_mot20/`.

In [ ]:
ds = Dataset()
fo_dataset = ds.prepare_all()

---
## 2. Train YOLO detector

Trains on MOT20-01 + MOT20-02, validates on MOT20-03.
Best weights are saved to `RUNS_DIR/train/weights/best.pt`.

In [ ]:
dataset_yaml = os.path.join(DATA_ROOT, "train_mot20", "dataset.yaml")

best_weights = train_detector(
    data_path=dataset_yaml,
    epochs=14,
    device=DEVICE,
    project=RUNS_DIR,
)
print(f"Best weights: {best_weights}")

---
## 3. Evaluate on validation set (MOT20-03)

Use this to tune hyperparameters (conf, iou thresholds).

In [ ]:
fo_dataset = fo.load_dataset("mot20")
model = __import__("ultralytics").YOLO(best_weights)

val_view = get_sequence_view(fo_dataset, "MOT20-03")
tracking_yolo(val_view, model, tag="val_pred", device=DEVICE)

print("\n── Validation metrics (MOT20-03) ──")
val_summary = calc_metrics(val_view, tag="val_pred")

---
## 4. Final evaluation on labeled test set (MOT20-05)

Run this **only once** after all tuning is complete.
This is the held-out sequence — do not use its results to change any model decisions.

In [ ]:
test_view = get_sequence_view(fo_dataset, "MOT20-05")
tracking_yolo(test_view, model, tag="test_pred", device=DEVICE)

print("\n── Test metrics (MOT20-05) ──")
test_summary = calc_metrics(test_view, tag="test_pred")

---
## 5. Visualise results in FiftyOne app

Opens an interactive browser session. Not available in Colab (use `save_results` instead).

In [ ]:
if ENVIRONMENT == "local":
    session = fo.launch_app(fo_dataset)
    # session.wait()  # uncomment to block until the app is closed
else:
    # Export annotated videos to Drive instead
    save_results(val_view,  "val_pred")
    save_results(test_view, "test_pred")